# 🔬 microGPT — The Entire LLM Algorithm in 200 Lines

**An interactive exploration of [Karpathy's microGPT](https://karpathy.github.io/2026/02/12/microgpt/)**

> *"This file contains the full algorithmic content of what is needed: dataset, tokenizer, autograd engine, a GPT-2-like neural network, the Adam optimizer, training loop, and inference loop. Everything else is just efficiency."* — Andrej Karpathy

This notebook breaks down every piece visually. By the end, you'll understand how a language model works **from raw text to generated output**, with zero magic hidden behind library calls.

---

### 🗺️ The Big Picture

```
┌─────────────────────────────────────────────────────────────────┐
│                    THE LLM PIPELINE                            │
│                                                                │
│  ┌──────────┐   ┌───────────┐   ┌─────────┐   ┌───────────┐  │
│  │  TEXT     │──▶│ TOKENIZER │──▶│  MODEL  │──▶│ NEXT TOKEN│  │
│  │ "emma"   │   │ [27,4,12] │   │  (GPT)  │   │ PROBS     │  │
│  └──────────┘   └───────────┘   └─────────┘   └───────────┘  │
│                                      │                        │
│                                 ┌────▼────┐                   │
│                                 │ AUTOGRAD│ (backward)        │
│                                 │ + ADAM  │ (update weights)  │
│                                 └─────────┘                   │
└─────────────────────────────────────────────────────────────────┘
```

That's it. Every LLM — from this 200-line script to ChatGPT — follows this exact pipeline. The difference is just scale and efficiency.

---

### 📚 Notebook Series

| # | Notebook | What you'll learn |
|---|----------|------------------|
| **01** | **microGPT Explained** (this one) | The complete algorithm, every piece |
| 02 | From microGPT → nanochat | How 200 lines scales to a real system |
| 03 | Training on Jetson | Edge AI: constraints, tricks, tradeoffs |
| 04 | Personalization via SFT | Teaching a model who you are |

---
## Part 1: The Dataset 📄

Every language model starts with text. ChatGPT uses the internet; we use 32,000 baby names.

The **core insight**: from the model's perspective, your conversation with ChatGPT is just a "document". When you type a prompt, the model sees it as the beginning of a document and tries to complete it in a way that's statistically consistent with its training data.

In [ ]:
import os, math, random
random.seed(42)

# Download the names dataset
if not os.path.exists('input.txt'):
    import urllib.request
    names_url = 'https://raw.githubusercontent.com/karpathy/makemore/988aa59/names.txt'
    urllib.request.urlretrieve(names_url, 'input.txt')

docs = [line.strip() for line in open('input.txt') if line.strip()]
random.shuffle(docs)

print(f"Total names: {len(docs):,}")
print(f"\nFirst 10 examples: {docs[:10]}")
print(f"\nShortest: '{min(docs, key=len)}' ({len(min(docs, key=len))} chars)")
print(f"Longest:  '{max(docs, key=len)}' ({len(max(docs, key=len))} chars)")

In [ ]:
# Let's visualize the length distribution
lengths = [len(d) for d in docs]
hist = {}
for l in lengths:
    hist[l] = hist.get(l, 0) + 1

print("Length distribution:")
print("─" * 50)
for length in sorted(hist.keys()):
    bar = '█' * (hist[length] // 100)
    print(f"  {length:2d} chars │ {bar} {hist[length]:,}")

---
## Part 2: The Tokenizer 🔤→🔢

Neural networks work with **numbers**, not letters. A tokenizer converts text ↔ integers.

```
"emma" → [4, 12, 12, 0]      (encode)
[4, 12, 12, 0] → "emma"      (decode)
```

Production tokenizers (like GPT-4's `tiktoken`) use **BPE** (Byte-Pair Encoding) to group common letter combinations into single tokens for efficiency. But the simplest tokenizer just maps each unique character to an integer.

**Key concept — BOS token**: We need a special "Beginning of Sequence" token to tell the model where documents start and end. Think of it as the model's period/paragraph marker.

In [ ]:
# Build the vocabulary: each unique character gets an integer ID
uchars = sorted(set(''.join(docs)))
BOS = len(uchars)  # Special token: Beginning of Sequence
vocab_size = len(uchars) + 1

print("Character → Token ID mapping:")
print("─" * 40)
for i, ch in enumerate(uchars):
    print(f"  '{ch}' → {i}", end="    ")
    if (i + 1) % 7 == 0: print()
print(f"\n  BOS → {BOS}")
print(f"\nVocab size: {vocab_size} (26 letters + 1 BOS)")

In [ ]:
# Let's see how a name gets tokenized
def encode(name):
    return [BOS] + [uchars.index(ch) for ch in name] + [BOS]

def decode(tokens):
    return ''.join(uchars[t] if t != BOS else '⟨BOS⟩' for t in tokens)

example = "emma"
tokens = encode(example)
print(f"Encoding '{example}':")
print(f"  Text:   {'  '.join(f' {c} ' for c in ['⟨BOS⟩'] + list(example) + ['⟨BOS⟩'])}")
print(f"  Tokens: {'  '.join(f'[{t:2d}]' for t in tokens)}")
print()
print("What the model sees during training:")
print("─" * 55)
print("  Given these tokens...     → predict this next token")
for i in range(len(tokens)-1):
    context = tokens[:i+1]
    target = tokens[i+1]
    ctx_str = ', '.join(str(t) for t in context)
    target_char = uchars[target] if target != BOS else '⟨BOS⟩'
    print(f"  [{ctx_str:20s}]  → {target} ('{target_char}')")

☝️ **This is the fundamental task of a language model**: given a sequence of tokens, predict the next one. That's it. Everything — from name generation to ChatGPT conversations — is just next-token prediction.

---
## Part 3: Autograd Engine ⛓️

Training needs **gradients**: "if I nudge this weight up, does the loss go up or down?"

The trick is the **chain rule** from calculus. Every computation is a chain of simple operations. Each operation knows its own local derivative. String them together backward from the loss, and you get gradients for everything.

```
Forward:   a ──[×]──▶ c ──[+]──▶ e ──▶ loss
              b ─┘       d ─┘

Backward:  ∂loss/∂a ◀── ∂loss/∂c ◀── ∂loss/∂e ◀── 1
              (chain rule applies local gradients at each step)
```

The `Value` class wraps a number and tracks the computation graph automatically:

In [ ]:
class Value:
    """A scalar value that tracks its computation graph for automatic differentiation."""
    __slots__ = ('data', 'grad', '_children', '_local_grads', '_op')

    def __init__(self, data, children=(), local_grads=(), op=''):
        self.data = data                # the actual number
        self.grad = 0                   # ∂loss/∂(this value), computed in backward()
        self._children = children       # inputs that produced this value
        self._local_grads = local_grads # ∂(this)/∂(each child)
        self._op = op                   # operation name (for visualization)

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a+b)/da = 1, d(a+b)/db = 1
        return Value(self.data + other.data, (self, other), (1, 1), '+')

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        # d(a*b)/da = b, d(a*b)/db = a  (the "swap" rule)
        return Value(self.data * other.data, (self, other), (other.data, self.data), '×')

    def __pow__(self, n):  return Value(self.data**n, (self,), (n * self.data**(n-1),), f'^{n}')
    def log(self):         return Value(math.log(self.data), (self,), (1/self.data,), 'log')
    def exp(self):         return Value(math.exp(self.data), (self,), (math.exp(self.data),), 'exp')
    def relu(self):        return Value(max(0, self.data), (self,), (float(self.data > 0),), 'relu')
    def __neg__(self):     return self * -1
    def __radd__(self, o): return self + o
    def __sub__(self, o):  return self + (-o)
    def __rsub__(self, o): return o + (-self)
    def __rmul__(self, o): return self * o
    def __truediv__(self, o):  return self * o**-1
    def __rtruediv__(self, o): return o * self**-1

    def backward(self):
        """Compute gradients for all nodes via reverse-mode autodiff (backpropagation)."""
        # Step 1: Build topological order (children before parents)
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        # Step 2: Walk backward, applying chain rule at each node
        self.grad = 1  # ∂loss/∂loss = 1
        for v in reversed(topo):
            for child, local_grad in zip(v._children, v._local_grads):
                child.grad += local_grad * v.grad  # THE chain rule

    def __repr__(self):
        return f"Value({self.data:.4f}, grad={self.grad:.4f})"

In [ ]:
# 🔬 Let's SEE the chain rule in action with a concrete example
# Compute: loss = (a * b + c)^2  where a=2, b=3, c=1
# Mathematical answer: d(loss)/da = 2*(a*b+c)*b = 2*7*3 = 42

a = Value(2.0); a._op = 'a'
b = Value(3.0); b._op = 'b'
c = Value(1.0); c._op = 'c'

# Forward pass (builds the graph)
d = a * b       # d = 6
e = d + c       # e = 7
loss = e ** 2   # loss = 49

# Backward pass (computes gradients)
loss.backward()

print("Forward pass (compute the result):")
print("─" * 50)
print(f"  a = {a.data}")
print(f"  b = {b.data}")
print(f"  c = {c.data}")
print(f"  d = a × b = {d.data}")
print(f"  e = d + c = {e.data}")
print(f"  loss = e² = {loss.data}")
print()
print("Backward pass (chain rule, right to left):")
print("─" * 50)
print(f"  ∂loss/∂loss = {loss.grad}           (start here, always 1)")
print(f"  ∂loss/∂e    = 2·e = {e.grad}         (power rule: d(e²)/de = 2e)")
print(f"  ∂loss/∂d    = {d.grad}              (addition passes gradient through)")
print(f"  ∂loss/∂c    = {c.grad}              (addition passes gradient through)")
print(f"  ∂loss/∂a    = {a.grad}              (multiply swaps: d(a·b)/da = b, then ×{e.grad})")
print(f"  ∂loss/∂b    = {b.grad}              (multiply swaps: d(a·b)/db = a, then ×{e.grad})")
print()
print("✓ Verify: ∂loss/∂a should be 2·(a·b+c)·b = 2·7·3 = 42.0 ✓" if a.grad == 42.0 else "✗ Error!")

In [ ]:
# 📊 Visualize the computation graph
def draw_graph(root, depth=0, prefix="", is_last=True):
    """ASCII art visualization of the computation graph."""
    connector = "└── " if is_last else "├── "
    if depth == 0:
        line = f"🎯 loss = {root.data:.1f}  (grad={root.grad:.1f})"
    else:
        op_str = f"[{root._op}]" if root._op else ""
        line = f"{prefix}{connector}{op_str} = {root.data:.1f}  (grad={root.grad:.1f})"
    print(line)
    extension = "    " if is_last else "│   "
    for i, child in enumerate(root._children):
        is_child_last = (i == len(root._children) - 1)
        draw_graph(child, depth+1, prefix + (extension if depth > 0 else ""), is_child_last)

print("Computation Graph (top = loss, leaves = inputs):")
print("═" * 55)
draw_graph(loss)
print()
print("The gradient flows BACKWARD (bottom → top) via chain rule.")
print("Each node's grad = sum of (parent's grad × local gradient) for all parents.")

### 🧠 Intuition Check

The gradient tells you: **"If I increase this value by a tiny amount ε, the loss changes by `grad × ε`"**

- `∂loss/∂a = 42` means: bump `a` up by 0.001 → loss goes up by 0.042
- During training, we want loss to go **down**, so we nudge `a` in the **opposite** direction of its gradient
- That's gradient descent! `a -= learning_rate × gradient`

---
## Part 4: The GPT Model Architecture 🏗️

Now the fun part. The GPT is a function: **tokens in → probability distribution over next token out**.

```
                    THE GPT ARCHITECTURE
                    ═══════════════════

token_id ──┐
            ├──▶ [Embedding] ──▶ x (vector of n_embd numbers)
pos_id ────┘         │
                     ▼
              ┌──────────────┐
              │  RMSNorm     │ (normalize the vector)
              └──────┬───────┘
                     │
         ┌───────────▼───────────┐
         │   TRANSFORMER BLOCK   │ ← repeat n_layer times
         │  ┌─────────────────┐  │
         │  │  Attention      │  │  "look at previous tokens"
         │  │  (Q, K, V)      │  │
         │  └────────┬────────┘  │
         │     + residual        │
         │  ┌────────▼────────┐  │
         │  │     MLP         │  │  "think about what you saw"
         │  │ (expand→relu→   │  │
         │  │  contract)      │  │
         │  └────────┬────────┘  │
         │     + residual        │
         └───────────┬───────────┘
                     │
                     ▼
              [lm_head linear]
                     │
                     ▼
              [softmax → probabilities]
                     │
                     ▼
         P(next_token = 'a') = 0.02
         P(next_token = 'b') = 0.01
         P(next_token = 'c') = 0.15  ← sample from this!
         ...
```

Let's build each piece.

### 4.1 Model Parameters (The Brain)

All the "knowledge" of the model lives in its **parameters** — just a big pile of numbers. Before training, these are random noise. After training, they encode patterns from the data.

In [ ]:
# Model hyperparameters
n_layer = 1       # depth: how many transformer blocks (stacked reasoning steps)
n_embd = 16       # width: dimension of internal representations
block_size = 16   # max context length (longest name is 15 chars)
n_head = 4        # number of attention heads (parallel "perspectives")
head_dim = n_embd // n_head  # each head works on this many dimensions

print(f"Model config:")
print(f"  n_layer = {n_layer}    (depth — more layers = more reasoning steps)")
print(f"  n_embd  = {n_embd}   (width — richer internal representations)")
print(f"  n_head  = {n_head}     (parallel attention perspectives)")
print(f"  head_dim = {head_dim}    (each head sees {head_dim} dimensions)")
print(f"  block_size = {block_size} (max sequence length)")

In [ ]:
# Initialize all parameters as random numbers wrapped in Value objects
matrix = lambda nout, nin, std=0.08: [[Value(random.gauss(0, std)) for _ in range(nin)] for _ in range(nout)]

state_dict = {
    'wte':     matrix(vocab_size, n_embd),   # token embeddings: 27 × 16
    'wpe':     matrix(block_size, n_embd),   # position embeddings: 16 × 16
    'lm_head': matrix(vocab_size, n_embd),   # output projection: 27 × 16
}
for i in range(n_layer):
    state_dict[f'layer{i}.attn_wq'] = matrix(n_embd, n_embd)    # 16×16
    state_dict[f'layer{i}.attn_wk'] = matrix(n_embd, n_embd)    # 16×16
    state_dict[f'layer{i}.attn_wv'] = matrix(n_embd, n_embd)    # 16×16
    state_dict[f'layer{i}.attn_wo'] = matrix(n_embd, n_embd)    # 16×16
    state_dict[f'layer{i}.mlp_fc1'] = matrix(4*n_embd, n_embd)  # 64×16 (expand)
    state_dict[f'layer{i}.mlp_fc2'] = matrix(n_embd, 4*n_embd)  # 16×64 (contract)

params = [p for mat in state_dict.values() for row in mat for p in row]

# Count parameters by component
print("Parameter count by component:")
print("─" * 50)
total = 0
for name, mat in state_dict.items():
    n = len(mat) * len(mat[0])
    total += n
    shape = f"{len(mat)}×{len(mat[0])}"
    bar = '█' * (n // 50)
    print(f"  {name:25s} {shape:8s} = {n:5d}  {bar}")
print(f"  {'TOTAL':25s}          = {total:5d}")
print(f"\nFor comparison: GPT-2 has 124M params, GPT-4 has ~1.8T params")

### 4.2 Building Blocks: Linear, Softmax, RMSNorm

In [ ]:
def linear(x, w):
    """Matrix-vector multiplication: each output is a weighted sum of inputs.
    
    x: [n_in] input vector
    w: [n_out × n_in] weight matrix
    returns: [n_out] output vector
    
    This is THE fundamental operation of neural networks.
    """
    return [sum(wi * xi for wi, xi in zip(wo, x)) for wo in w]

def softmax(logits):
    """Convert raw scores into a probability distribution (sums to 1).
    
    Higher scores → higher probabilities, but normalized.
    The max subtraction is for numerical stability (doesn't change the result).
    """
    max_val = max(val.data for val in logits)
    exps = [(val - max_val).exp() for val in logits]
    total = sum(exps)
    return [e / total for e in exps]

def rmsnorm(x):
    """Normalize a vector to have unit root-mean-square magnitude.
    
    Without normalization, values can grow or shrink uncontrollably
    as they pass through many layers. This keeps things stable.
    """
    ms = sum(xi * xi for xi in x) / len(x)
    scale = (ms + 1e-5) ** -0.5
    return [xi * scale for xi in x]

# 🔬 Demo: softmax in action
print("Softmax demo — converting scores to probabilities:")
print("─" * 50)
scores = [Value(2.0), Value(1.0), Value(0.1), Value(-1.0), Value(3.0)]
probs = softmax(scores)
labels = ['a', 'b', 'c', 'd', 'e']
for label, score, prob in zip(labels, scores, probs):
    bar = '█' * int(prob.data * 40)
    print(f"  '{label}': score={score.data:5.1f} → prob={prob.data:.3f} {bar}")
print(f"  Sum of probs: {sum(p.data for p in probs):.4f} (should be 1.0)")

### 4.3 The Attention Mechanism 👀

**Attention is the core innovation of the Transformer.** It lets each token "look at" all previous tokens and decide which ones are relevant.

The analogy: imagine you're reading the name "chris" and you're at the letter 'i'. To predict what comes next, you might want to look back at 'c', 'h', 'r' to understand the pattern. Attention does exactly this — it computes a weighted average of all previous tokens, where the weights are learned.

```
For each token position:
  1. Compute Query (Q): "What am I looking for?"
  2. Compute Key (K):   "What do I contain?" (for all previous tokens)
  3. Compute Value (V): "What information should I provide?"
  4. Attention = softmax(Q · K^T / √d) × V
     "How much should I attend to each previous token?"
```

In [ ]:
# Let's trace attention step by step for a tiny example
# Imagine we have 3 tokens processed so far, and we're computing attention for the 3rd

print("Attention step-by-step for 'e', 'm', 'm' (positions 0, 1, 2):")
print("═" * 60)
print()
print("Step 1: Each token computes Q, K, V vectors")
print("─" * 60)
print("  Token 'e' at pos 0: Q₀=[...], K₀=[...], V₀=[...]")
print("  Token 'm' at pos 1: Q₁=[...], K₁=[...], V₁=[...]")
print("  Token 'm' at pos 2: Q₂=[...], K₂=[...], V₂=[...]")
print()
print("Step 2: Token at pos 2 ('m') computes attention scores")
print("─" * 60)
print("  score(2→0) = Q₂ · K₀ / √d  (how much should 'm' attend to 'e'?)")
print("  score(2→1) = Q₂ · K₁ / √d  (how much should 'm' attend to 'm'?)")
print("  score(2→2) = Q₂ · K₂ / √d  (how much should 'm' attend to itself?)")
print()
print("Step 3: Softmax → attention weights (probability distribution)")
print("─" * 60)
print("  weights = softmax([score₀, score₁, score₂])")
print("  e.g., [0.1, 0.6, 0.3] means 'm' mostly looks at the previous 'm'")
print()
print("Step 4: Weighted sum of Values")
print("─" * 60)
print("  output = 0.1·V₀ + 0.6·V₁ + 0.3·V₂")
print("  This is the 'context-aware' representation of position 2")
print()
print("💡 KEY INSIGHT: Attention is CAUSAL — token 2 can only look at")
print("   tokens 0, 1, 2 (not future tokens). This is what makes GPT")
print("   autoregressive: it generates one token at a time, left to right.")

In [ ]:
# Multi-head attention: run n_head parallel attention operations
# Each head looks at a different n_embd/n_head slice of the embedding

print(f"Multi-head attention with {n_head} heads:")
print("═" * 60)
print(f"  Full embedding dimension: {n_embd}")
print(f"  Each head sees: {head_dim} dimensions")
print()
for h in range(n_head):
    start = h * head_dim
    end = start + head_dim
    dims = f"dims[{start}:{end}]"
    print(f"  Head {h}: {dims}  — might learn to focus on different patterns")
    print(f"           e.g., head 0: 'what letter came before?'")
    print(f"                 head 1: 'is this a vowel or consonant?'")
    if h == 1: print(f"                 ... (the model learns what to focus on)")
    if h > 1: break
print(f"\n  Results from all heads are concatenated: {n_head}×{head_dim} = {n_embd}")
print(f"  Then projected back through Wo to get the final attention output")

### 4.4 The Complete GPT Forward Pass

Now let's put it all together. The `gpt()` function processes **one token at a time** (autoregressive), using a KV cache to store previous tokens' keys and values.

In [ ]:
def gpt(token_id, pos_id, keys, values, verbose=False):
    """Process one token through the GPT, return logits for next token."""
    
    # Step 1: Embeddings — look up the token and position vectors
    tok_emb = state_dict['wte'][token_id]  # [n_embd] — "what token is this?"
    pos_emb = state_dict['wpe'][pos_id]    # [n_embd] — "where in the sequence?"
    x = [t + p for t, p in zip(tok_emb, pos_emb)]  # combine both
    x = rmsnorm(x)
    
    if verbose:
        print(f"  Embedding: tok[{token_id}] + pos[{pos_id}] → vector of {len(x)} numbers")

    for li in range(n_layer):
        # ═══ ATTENTION BLOCK ═══
        x_residual = x  # save for residual connection
        x = rmsnorm(x)
        
        # Project to Q, K, V
        q = linear(x, state_dict[f'layer{li}.attn_wq'])
        k = linear(x, state_dict[f'layer{li}.attn_wk'])
        v = linear(x, state_dict[f'layer{li}.attn_wv'])
        
        # Cache K, V for future tokens to attend to
        keys[li].append(k)
        values[li].append(v)
        
        # Multi-head attention
        x_attn = []
        for h in range(n_head):
            hs = h * head_dim
            q_h = q[hs:hs+head_dim]
            k_h = [ki[hs:hs+head_dim] for ki in keys[li]]
            v_h = [vi[hs:hs+head_dim] for vi in values[li]]
            # Attention scores: Q · K^T / sqrt(head_dim)
            attn_logits = [
                sum(q_h[j] * k_h[t][j] for j in range(head_dim)) / head_dim**0.5
                for t in range(len(k_h))
            ]
            attn_weights = softmax(attn_logits)
            # Weighted sum of values
            head_out = [
                sum(attn_weights[t] * v_h[t][j] for t in range(len(v_h)))
                for j in range(head_dim)
            ]
            x_attn.extend(head_out)
        
        x = linear(x_attn, state_dict[f'layer{li}.attn_wo'])
        x = [a + b for a, b in zip(x, x_residual)]  # residual connection
        
        if verbose:
            print(f"  After attention: attended to {len(keys[li])} positions")
        
        # ═══ MLP BLOCK ═══
        x_residual = x
        x = rmsnorm(x)
        x = linear(x, state_dict[f'layer{li}.mlp_fc1'])  # expand: 16 → 64
        x = [xi.relu() for xi in x]                       # nonlinearity
        x = linear(x, state_dict[f'layer{li}.mlp_fc2'])  # contract: 64 → 16
        x = [a + b for a, b in zip(x, x_residual)]       # residual connection
        
        if verbose:
            print(f"  After MLP: 16→64→ReLU→16 (expand, think, compress)")
    
    # Output: project to vocabulary size
    logits = linear(x, state_dict['lm_head'])
    
    if verbose:
        print(f"  Output: {len(logits)} logits (one score per vocab token)")
    
    return logits

In [ ]:
# 🔬 Trace a forward pass on "emma"
print("Tracing GPT forward pass on 'emma':")
print("═" * 60)

tokens = encode("emma")
keys_trace, values_trace = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]

for pos in range(len(tokens) - 1):
    tok = tokens[pos]
    target = tokens[pos + 1]
    tok_str = uchars[tok] if tok != BOS else '⟨BOS⟩'
    tgt_str = uchars[target] if target != BOS else '⟨BOS⟩'
    print(f"\nPosition {pos}: input='{tok_str}' (token {tok}), target='{tgt_str}' (token {target})")
    print("─" * 60)
    logits = gpt(tok, pos, keys_trace, values_trace, verbose=True)
    probs = softmax(logits)
    target_prob = probs[target].data
    # Show top 5 predictions
    prob_list = [(p.data, i) for i, p in enumerate(probs)]
    prob_list.sort(reverse=True)
    print(f"  Top predictions:")
    for prob, idx in prob_list[:5]:
        ch = uchars[idx] if idx != BOS else '⟨BOS⟩'
        marker = " ◀ target" if idx == target else ""
        print(f"    P('{ch}') = {prob:.3f}{marker}")
    print(f"  Loss for this position: -log({target_prob:.4f}) = {-math.log(target_prob):.2f}")

print(f"\n💡 Before training, predictions are ~random (~1/{vocab_size} = {1/vocab_size:.3f} per token)")
print(f"   Expected random loss: -log(1/{vocab_size}) = {math.log(vocab_size):.2f}")

---
## Part 5: Training Loop 🏋️

Training is deceptively simple:

```
for each training step:
    1. Pick a name from the dataset
    2. Forward: run it through the model, compute loss
    3. Backward: compute gradients for all parameters
    4. Update: nudge each parameter to reduce the loss (Adam optimizer)
```

The **loss** measures how surprised the model is by the actual next token. If the model predicted P('m')=0.9 and the actual next token is 'm', the loss is low (-log(0.9)=0.1). If P('m')=0.01, the loss is high (-log(0.01)=4.6).

In [ ]:
# Adam optimizer buffers
learning_rate, beta1, beta2, eps_adam = 0.01, 0.85, 0.99, 1e-8
m_buf = [0.0] * len(params)  # first moment (running average of gradients)
v_buf = [0.0] * len(params)  # second moment (running average of squared gradients)

print("Adam optimizer:")
print("─" * 50)
print(f"  learning_rate = {learning_rate}  (step size)")
print(f"  beta1 = {beta1}  (momentum — 'keep going in the same direction')")
print(f"  beta2 = {beta2}  (variance — 'scale steps by gradient history')")
print(f"  {len(params)} parameters to optimize")
print()
print("Why Adam over plain gradient descent?")
print("  SGD:  param -= lr × gradient  (can oscillate or get stuck)")
print("  Adam: keeps a running average of past gradients (momentum)")
print("        AND adapts step size per-parameter (some need big steps,")
print("        others need tiny steps). Almost always better.")

In [ ]:
# 🚀 TRAINING! This takes ~2-5 minutes for 1000 steps
num_steps = 1000
loss_history = []

print(f"Training for {num_steps} steps...")
print("═" * 60)

for step in range(num_steps):
    # 1. Get a training example
    doc = docs[step % len(docs)]
    tokens = [BOS] + [uchars.index(ch) for ch in doc] + [BOS]
    n = min(block_size, len(tokens) - 1)

    # 2. Forward pass: build computation graph, compute loss
    keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
    losses = []
    for pos_id in range(n):
        token_id, target_id = tokens[pos_id], tokens[pos_id + 1]
        logits = gpt(token_id, pos_id, keys, values)
        probs = softmax(logits)
        loss_t = -probs[target_id].log()
        losses.append(loss_t)
    loss = (1 / n) * sum(losses)

    # 3. Backward pass: compute gradients
    loss.backward()

    # 4. Adam optimizer update
    lr_t = learning_rate * (1 - step / num_steps)  # linear LR decay
    for i, p in enumerate(params):
        m_buf[i] = beta1 * m_buf[i] + (1 - beta1) * p.grad
        v_buf[i] = beta2 * v_buf[i] + (1 - beta2) * p.grad ** 2
        m_hat = m_buf[i] / (1 - beta1 ** (step + 1))
        v_hat = v_buf[i] / (1 - beta2 ** (step + 1))
        p.data -= lr_t * m_hat / (v_hat ** 0.5 + eps_adam)
        p.grad = 0  # reset gradient for next step

    loss_history.append(loss.data)
    if (step + 1) % 100 == 0 or step == 0:
        bar = '█' * int(40 * (1 - loss.data / 4))
        print(f"  step {step+1:4d}/{num_steps} | loss {loss.data:.4f} | lr {lr_t:.4f} | {bar}")

print(f"\nTraining complete! Final loss: {loss_history[-1]:.4f}")
print(f"Random baseline loss: {math.log(vocab_size):.4f}")
print(f"Improvement: {math.log(vocab_size)/loss_history[-1]:.1f}× better than random")

In [ ]:
# 📊 Plot the loss curve (ASCII art)
print("\nTraining Loss Curve:")
print("═" * 62)

# Bucket into 50 bins
n_bins = 50
bin_size = len(loss_history) // n_bins
binned = [sum(loss_history[i*bin_size:(i+1)*bin_size]) / bin_size for i in range(n_bins)]

max_loss = max(binned)
min_loss = min(binned)
height = 15

for row in range(height, -1, -1):
    threshold = min_loss + (max_loss - min_loss) * row / height
    label = f"{threshold:.2f}" if row % 3 == 0 else "    "
    line = f"{label:>5s} │"
    for val in binned:
        line += '█' if val >= threshold else ' '
    print(line)

print(f"      └{'─' * n_bins}")
print(f"       step 1{' ' * (n_bins - 10)}step {num_steps}")
print(f"\n  Loss drops from {loss_history[0]:.2f} → {loss_history[-1]:.2f}")
print(f"  The model is LEARNING! It's getting less surprised by the data.")

---
## Part 6: Inference — Generate New Names! 🎲

Now the model has learned patterns from the names. Let's generate new ones!

```
Inference loop:
  1. Start with BOS token
  2. Run through GPT → get probabilities
  3. Sample next token from the distribution
  4. If BOS again → done (end of name)
  5. Otherwise → append token and go to step 2
```

**Temperature** controls creativity:
- Low (0.1): always picks the most likely token → boring but safe
- High (1.0+): more random sampling → creative but potentially weird

In [ ]:
def generate(temperature=0.5, n_samples=20):
    """Generate new names from the trained model."""
    samples = []
    for _ in range(n_samples):
        keys, values = [[] for _ in range(n_layer)], [[] for _ in range(n_layer)]
        token_id = BOS
        name = []
        for pos_id in range(block_size):
            logits = gpt(token_id, pos_id, keys, values)
            probs = softmax([l / temperature for l in logits])
            token_id = random.choices(range(vocab_size), weights=[p.data for p in probs])[0]
            if token_id == BOS:
                break
            name.append(uchars[token_id])
        samples.append(''.join(name))
    return samples

print("🎲 Generated names at different temperatures:")
print("═" * 60)
for temp in [0.3, 0.5, 0.8, 1.2]:
    names = generate(temperature=temp, n_samples=10)
    creativity = {0.3: '🧊 Conservative', 0.5: '⚖️ Balanced', 0.8: '🎨 Creative', 1.2: '🌪️ Wild'}[temp]
    print(f"\n  Temperature {temp} ({creativity}):")
    print(f"    {', '.join(names)}")

---
## Part 7: The Big Picture — microGPT → ChatGPT 🔭

Everything in this notebook is **the actual algorithm**. ChatGPT uses the exact same pieces, just bigger and optimized:

| Component | microGPT (this) | nanochat (our Jetson) | GPT-4 |
|-----------|-----------------|----------------------|-------|
| **Parameters** | 3,563 | 36.7M | ~1.8T |
| **Layers** | 1 | 4 | ~120 |
| **Embedding dim** | 16 | 256 | ~12,288 |
| **Context length** | 16 | 1,024 | 128K |
| **Vocab size** | 27 | 32,768 | ~100K |
| **Training data** | 32K names | ClimbMix (web text) | Internet |
| **Training time** | ~3 min (CPU) | ~20 min (Jetson) | Months (thousands of GPUs) |
| **Autograd** | `Value` class | PyTorch | PyTorch |
| **Tokenizer** | char-level | BPE (32K vocab) | BPE (100K vocab) |

### What "Everything else is just efficiency" means:

| Optimization | What it does | Present in microGPT? |
|-------------|-------------|---------------------|
| **GPU/CUDA** | Parallel matrix math | ❌ (pure Python loops) |
| **Batching** | Process multiple examples at once | ❌ (one name at a time) |
| **torch.compile** | Fuse operations, eliminate overhead | ❌ |
| **Flash Attention** | Memory-efficient attention | ❌ |
| **Mixed precision (bf16)** | Half-precision math for 2× speed | ❌ |
| **Gradient accumulation** | Simulate larger batches | ❌ |
| **BPE tokenizer** | Fewer tokens per document | ❌ (char-level) |
| **KV cache** | Don't recompute past tokens | ✅ (keys/values lists) |
| **Distributed training** | Multiple GPUs | ❌ |

Every single one of these is "just" making the same algorithm run faster or on more data. The algorithm itself — forward, backward, update — is exactly what you see in this notebook.

---
## 🧪 Exercises (try these!)

1. **Change the data**: Replace `input.txt` with a different dataset (city names? words from a dictionary?). Does the model learn different patterns?

2. **Scale up**: Try `n_layer=2`, `n_embd=32`. How does loss change? How much slower is training?

3. **Inspect attention**: After training, print the attention weights for a name like "chris". Which previous tokens does the model attend to most?

4. **Break it**: Set `learning_rate=1.0`. What happens? Why?

5. **Connect to nanochat**: Open `nanochat/gpt.py` in our repo and find the same components: embedding, attention, MLP, residual connections. The architecture is identical, just written in PyTorch instead of from-scratch.

---

**Next notebook**: `02_micro_to_nano.ipynb` — How these 200 lines scale to the nanochat system we trained on Jetson.